# STAR-v3 — Train 10K-hard tren Kaggle (X-VLM 16M + LoRA + ViTPose + LHP)

**Cai dat notebook (BAT BUOC truoc khi Run):**
- Settings → Accelerator = **GPU T4 x2** (hoac T4) · Internet = **ON**
- **Add Input** → dataset cua ban gom 2 file: `train_10k_hard_data.tar.zst` (1.6GB) va `xvlm_16m_base.th` (825MB)

**Pipeline:** clone repo → env pinned X-VLM → giai nen data → build manifest tu `train_10k_hard.jsonl` **+ ViTPose enrichment** (keypoints 17x3 cho pose branch, bbox cho LHP person-crop) → gate overfit → train LoRA → evaluate mAP.

**Cau hinh run nay (theo plan):** text encoder FROZEN · LoRA image+cross · **pose branch ON** (keypoint → MLP → gated-fuse vao f_V, train + eval deu fuse) · **LHP ON** (crop quanh bbox nguoi, scale≥0.5; inference dung anh full) · `L = ITC + 1.0·ITM(hard-neg) + 0.2·Smooth-AP`.

**Du lieu:** 11,735 cap (10,000 anchor + 1,735 hard-only) · train 10,149 · VAL-B 621 query + 965 distractor · coverage keypoints/bbox = **100%** · leakage video = 0 *(tat ca da kiem tren data that)*.

**Thoi gian uoc tinh (T4, batch 16 @384, fp16):** ~10–16 phut/epoch (634 step; pose branch +<1% compute, LHP chi la crop trong dataloader) → early-stop 3–5 epoch → **tong ~1–2h**. VRAM du tinh ~10–13GB / 16GB (neu OOM: ha batch=12).

In [ ]:
# [1/7] GPU check + clone repo
import os, pathlib, subprocess
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip()
assert gpu, "KHONG THAY GPU — bat Settings > Accelerator = GPU T4 roi chay lai!"
print("GPU:", gpu)

os.chdir("/kaggle/working")
if not pathlib.Path("star").exists():
    !git clone -q https://github.com/Khanhhh239/Model_XVLM_Training.git star
os.chdir("/kaggle/working/star")
print("repo:", os.getcwd())

In [ ]:
# [2/7] Env pinned cho X-VLM (transformers 4.12.5; GIU torch CUDA cua Kaggle) + 2 patch + X-VLM source
!pip -q install "numpy<2" "scipy==1.11.4" "huggingface_hub==0.10.1" "tokenizers>=0.13,<0.14" sacremoses ruamel.yaml gdown pyarrow zstandard
!pip -q install --no-deps "transformers==4.12.5" "timm==0.4.9"

# patch 1: noi long pin tokenizers cua transformers (X-VLM dung tokenizer python rieng)
import importlib.util, pathlib, re
spec = importlib.util.find_spec("transformers")
f = pathlib.Path(spec.origin).parent / "dependency_versions_table.py"
f.write_text(re.sub(r'"tokenizers":[^,]+', '"tokenizers": "tokenizers"', f.read_text()))

# X-VLM source + patch 2: CIDEr (captioning, khong dung) -> optional
if not pathlib.Path("third_party/X-VLM").exists():
    !git clone -q --depth 1 https://github.com/zengyan-97/X-VLM third_party/X-VLM
u = pathlib.Path("third_party/X-VLM/utils/__init__.py"); s = u.read_text()
old = "from utils.cider.pyciderevalcap.ciderD.ciderD import CiderD"
if "try:\n    " + old not in s:
    u.write_text(s.replace(old, "try:\n    " + old + "\nexcept Exception:\n    CiderD = None"))
print("env + X-VLM source READY")

In [ ]:
# [3/7] Tim dataset trong /kaggle/input: checkpoint + data (tar.zst HOAC folder da giai nen)
import glob, os, pathlib

def find_one(pattern):
    hits = sorted(set(glob.glob(f"/kaggle/input/*/{pattern}") +
                      glob.glob(f"/kaggle/input/*/**/{pattern}", recursive=True)))
    return hits[0] if hits else None

CKPT  = find_one("xvlm_16m_base.th")
JSONL = find_one("train_10k_hard.jsonl")              # co neu upload folder da giai nen
ARCH  = find_one("train_10k_hard_data.tar.zst")        # co neu upload file nen (khuyen nghi)

if JSONL:
    DATA_ROOT = str(pathlib.Path(JSONL).parents[2])    # .../train_10k_hard_data
elif ARCH:
    marker = "/kaggle/working/extract"
    got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    if not got:
        os.makedirs(marker, exist_ok=True)
        print("extracting 1.6GB (~2-4 phut)...")
        rc = os.system(f"tar -I 'zstd -d' -xf '{ARCH}' -C {marker}")
        if rc != 0:                                     # fallback python neu thieu zstd binary
            import zstandard, tarfile
            with open(ARCH, "rb") as fh, zstandard.ZstdDecompressor().stream_reader(fh) as zr:
                with tarfile.open(fileobj=zr, mode="r|") as tf:
                    tf.extractall(marker)
        got = glob.glob(f"{marker}/**/train_10k_hard.jsonl", recursive=True)
    DATA_ROOT = str(pathlib.Path(got[0]).parents[2])
else:
    raise FileNotFoundError("Khong thay data! Add Input dataset chua train_10k_hard_data.tar.zst")

if not CKPT:                                            # fallback: tai checkpoint qua gdown
    print("checkpoint khong co trong dataset -> tai gdown (825MB)...")
    os.makedirs("data/checkpoints", exist_ok=True)
    !gdown -q 1iXgITaSbQ1oGPPvGaV0Hlae4QiJG5gx0 -O data/checkpoints/xvlm_16m_base.th
    CKPT = "data/checkpoints/xvlm_16m_base.th"

assert pathlib.Path(CKPT).stat().st_size > 8e8, "checkpoint hong/thieu (can ~825MB)"
print("DATA_ROOT =", DATA_ROOT)
print("CKPT      =", CKPT)

In [ ]:
# [4/7] Build manifest tu train_10k_hard.jsonl + ENRICH ViTPose (keypoints + bbox)
#  - anchor (image_webp, caption) + hard image (hard_image_webp, hard_c) = cac cap train
#    (cung scene-group 'v{video_id}' -> sampler gom anchor + hard pair vao cung batch)
#  - sequence_id = video_id + bucket: cung video+bucket = positive, khac bucket = negative
#  - keypoints: instances[0].keypoints_xyc (17x[x,y,c]) -> 51 floats, x/y chia W/H (chuan hoa)
#  - bbox cho LHP: primary_bbox_norm_xyxy -> [x, y, w, h] chuan hoa
#  - split theo VIDEO: ~600 query + ~900 distractor (caption rong) -> khong leakage
#  (logic DA KIEM TREN DATA THAT: 11,735 rows, coverage keypoints/bbox = 100%, leakage = 0)
import json, pathlib
import numpy as np, pandas as pd

root = pathlib.Path(DATA_ROOT)
anchors, hard_rows, anchor_ids = [], {}, set()
def bucket_of(p):
    return "wentwrong" if "/wentwrong/" in p else ("full" if "/full/" in p else "goal")

for line in open(root / "data/subsets/train_10k_hard.jsonl", encoding="utf-8"):
    r = json.loads(line)
    anchor_ids.add(r["image_id"])
    anchors.append(dict(image_path=r["image_webp"], caption=r["caption"],
        sequence_id=f'v{r["video_id"]}_{r["bucket"]}', scene=f'v{r["video_id"]}',
        action=str(r.get("label", "unk")), video_id=r["video_id"], image_id=r["image_id"]))
    hid = r.get("hard_i_id")
    if hid and hid not in hard_rows:
        hard_rows[hid] = dict(image_path=r["hard_image_webp"], caption=r["hard_c"],
            sequence_id=f'v{r["video_id"]}_{bucket_of(r["hard_image_webp"])}', scene=f'v{r["video_id"]}',
            action="hard_pair", video_id=r["video_id"], image_id=hid)

extra = [v for k, v in hard_rows.items() if k not in anchor_ids]
df = pd.DataFrame(anchors + extra)

# ---- ViTPose enrichment (team data da trich san) ----
pose = json.load(open(root / "data/subsets/prepared/train_10k_hard_vitpose.json", encoding="utf-8"))["items"]
def kpts_of(iid):
    it = pose.get(iid)
    if not it or it.get("status") != "ok" or not it.get("instances"):
        return None
    W, H = it.get("width", 384), it.get("height", 384)
    flat = []
    for x, y, c in it["instances"][0]["keypoints_xyc"]:
        flat += [x / W, y / H, c]
    return flat if len(flat) == 51 else None
def bbox_of(iid):
    it = pose.get(iid)
    b = it.get("primary_bbox_norm_xyxy") if it else None
    if not b:
        return None
    x1, y1, x2, y2 = b
    return [x1, y1, max(x2 - x1, 1e-3), max(y2 - y1, 1e-3)]
df["keypoints"] = df.image_id.map(kpts_of)
df["bbox"] = df.image_id.map(bbox_of)
cov_k, cov_b = df.keypoints.notna().mean(), df.bbox.notna().mean()

# ---- split theo video ----
rng = np.random.default_rng(42)
vids = df.video_id.unique().copy(); rng.shuffle(vids)
counts = df.groupby("video_id").size(); vq, vd, acc = set(), set(), 0
it = iter(vids)
for v in it:
    vq.add(v); acc += counts[v]
    if acc >= 600: break
acc = 0
for v in it:
    vd.add(v); acc += counts[v]
    if acc >= 900: break
df["split"] = np.where(df.video_id.isin(vq | vd), "valb", "train")
df.loc[df.video_id.isin(vd), "caption"] = ""          # distractor: anh-only trong gallery VAL-B

MANIFEST = "/kaggle/working/manifest_10k_hard.parquet"
df.to_parquet(MANIFEST, index=False)

n_q = ((df.split == "valb") & (df.caption != "")).sum()
n_d = ((df.split == "valb") & (df.caption == "")).sum()
leak = len(set(df[df.split == "train"].video_id) & set(df[df.split == "valb"].video_id))
miss = sum(1 for p in df.image_path.sample(200, random_state=0) if not (root / p).exists())
print(f"rows={len(df)}  train={(df.split=='train').sum()}  valb_query={n_q}  valb_distractor={n_d}")
print(f"coverage: keypoints={cov_k:.1%}  bbox={cov_b:.1%}  (can ~100%)")
print(f"video leakage={leak} (phai=0)   missing files/200={miss} (phai=0)")
assert leak == 0 and miss == 0 and cov_k > 0.99 and cov_b > 0.99
print("MANIFEST =", MANIFEST)

In [ ]:
# [5/7] GATE: overfit-one-batch (~5-8 phut: build model + load ckpt + 300 step)
# Loss PHAI giam >=75% -> wiring dung -> moi chay full. Neu OOM: them " train.batch_size=12" vao SET.
SET = (f"data.manifest={MANIFEST} data.image_root={DATA_ROOT} model.checkpoint={CKPT} "
       f"data.lhp_enabled=true model.pose_enabled=true loss.lambda_smooth_ap=0.2")
cmd = f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml --overfit-one-batch --set {SET}"
print(cmd)
!{cmd}

In [ ]:
# [6/7] TRAIN FULL — 634 step/epoch, eval VAL-B moi nua epoch, early-stop patience 2, luu best.pth
import time
t0 = time.time()
cmd = (f"python scripts/train.py --config configs/star_v3_10k_kaggle.yaml "
       f"--set {SET} train.out_dir=/kaggle/working/run10k")
print(cmd)
!{cmd}
print(f"TOTAL TRAIN: {(time.time() - t0) / 60:.1f} phut")

In [ ]:
# [7/7] EVALUATE best.pth tren VAL-B (621 query vs gallery ~1,580 anh trong do 965 distractor)
cmd = (f"python scripts/evaluate.py --config configs/star_v3_10k_kaggle.yaml "
       f"--ckpt /kaggle/working/run10k/best.pth --set {SET}")
print(cmd)
!{cmd}
!ls -lh /kaggle/working/run10k/

## Sau khi chay xong
- **Tai checkpoint:** panel phai → Output → `run10k/best.pth` (~860MB). Muon luu vinh vien: **Save Version → Save & Run All** roi lay tu tab Output cua version.
- **Diem can xem:** dong `[VAL-B] {'mAP': ...}` — mAP tren gallery CO 965 distractor. Pose duoc fuse o CA train va eval (train/eval nhat quan).
- **Luu y khi inference sau nay:** model nay can keypoints cho moi anh gallery (pose branch ON) va dung anh FULL (khong LHP).

## Troubleshooting
| Trieu chung | Cach xu ly |
|---|---|
| `CUDA out of memory` | them ` train.batch_size=12` (hoac 8) vao SET o cell 5 roi chay lai 5→7 |
| gdown loi quota (neu thieu ckpt trong dataset) | tai `xvlm_16m_base.th` ve may → upload vao dataset → chay lai cell 3 |
| `ImportError` trong cell train (transformers/xbert) | gui stack trace — to hop transformers 4.12.5 + torch Kaggle chua test that tren Kaggle |
| Loss `nan` | guard tu skip step; neu lap lai nhieu → them ` optim.lr_lora=1e-4` vao SET |
| coverage keypoints < 100% | van chay duoc (batch thieu kpts se bo fuse pose); bao lai de kiem tra vitpose.json |

## Buoc tiep theo (sau run dau)
- So sanh voi run pose/LHP OFF: bo `data.lhp_enabled=true model.pose_enabled=true` khoi SET → biet 2 nhanh nay co giup that khong (quyet dinh bang VAL-B, khong doan).
- Sweep `loss.lambda_smooth_ap` ∈ {0, 0.1, 0.2, 0.3} va `optim.lr_lora` ∈ {5e-5, 1e-4, 2e-4}.
- `train.grad_norm_every=200` de xem ti le gradient ITC/ITM/SmoothAP.